<a href="https://colab.research.google.com/github/Ramprasath-Meganathan/LoRA-Plus/blob/main/ML_Project_LoRA%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Master toggles - Change to True ONLY when necessary
RUN_INSTALLATION = False
RUN_HEAVY_TRAINING = False
DOWNLOAD_DATASETS = False

In [ ]:
# Upgrade torchao to meet the requirements of the updated PEFT library
!pip install -U -q torchao

In [ ]:
# Install required libraries
try:
  import transformers
  print("Libraries already installed! Skipping Installation.")
except ImportError:
  print("Libraries not found. Installing now...")
  !pip install -U -q torch transformers datasets peft scikit-learn matplotlib

Libraries already installed! Skipping Installation.


In [ ]:
!pip install -U "huggingface_hub<0.26.0"

  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.1.0 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 0.25.2 which is incompatible.
transformers 5.22.1 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 0.25.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.25.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.25.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.25.2 which is incompatible.


In [ ]:
# Necessary Library Imports
import os
import math
import time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import transformers
import datasets
import peft
from datasets import load_dataset
from torch.optim import AdamW
from google.colab import drive
from peft import get_peft_model, LoraConfig, TaskType
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [ ]:
print("PyTorch", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)

PyTorch 2.11.0+cu130
Transformers: 5.21.0
Datasets: 4.8.5
PEFT: 0.21.2


In [ ]:
# Verify GPU environment (Google Colab T4 16GB limit)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name: Tesla T4


In [ ]:
# Mount Google Drive for saving checkpoints to protect against session timeouts
drive.mount('/content/drive')
CHECKPOINT_DIR = "/content/drive/MyDrive/LoRA_Plus_Checkpoints/"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoints will be saved to: /content/drive/MyDrive/LoRA_Plus_Checkpoints/


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load the dataset
dataset = load_dataset("nyu-mll/glue", "sst2")

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize_function(batch):
  return tokenizer(
      batch["sentence"],
      truncation=True,
      max_length=128
  )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True)

tokenized_dataset = tokenized_dataset.remove_columns(
    ["sentence", "idx"]
)

tokenized_dataset = tokenized_dataset.rename_column(
    "label", "labels"
)

tokenized_dataset.set_format("torch")

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  148kB            

... (8 outputs left out)

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=["query","value"],
    modules_to_save=["classifier"],
    bias="none"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066
